# Project 006: Naive Bayes Classifier for Spam & Phishing Detection

**Category:** Machine Learning Basic Projects  
**Domain:** Foundational Natural Language Processing & Security  
**Architecture:** Multinomial, Bernoulli & Complement Naive Bayes with Laplace/Lidstone Smoothing  

---

### 📖 Theoretical Foundation

Naive Bayes is a supervised probabilistic classification model built on **Bayes' Theorem** with the foundational **conditional independence assumption**: every word in a document is assumed conditionally independent of every other word given the message class.

$$
P(Y = c \mid X) = \frac{P(Y = c) \prod_{i=1}^n P(x_i \mid Y = c)}{P(X)}
$$

To avoid floating-point underflow when multiplying hundreds of small probabilities, calculations are performed in logarithmic space:

$$
\log P(Y = c \mid X) \propto \log P(Y = c) + \sum_{i=1}^n \log P(x_i \mid Y = c)
$$

To solve the zero-frequency problem where unseen tokens cause the entire probability product to become zero, we apply **Laplace/Lidstone Smoothing ($\alpha$)**:

$$
\hat{\theta}_{ci} = \frac{N_{ci} + \alpha}{N_c + \alpha \cdot |V|}
$$

---

### 🎯 Notebook Workflow
1. **Environment Setup & Imports**: Load scikit-learn, pandas, numpy, and project modules.
2. **Dataset Synthesis & Inspection**: Load 1,200 curated spam, phishing, and legitimate messages.
3. **Stratified Partitioning**: Split into training (75%) and holdout test (25%) sets.
4. **Model Architecture Benchmarking**: Compare MultinomialNB, ComplementNB, and BernoulliNB.
5. **Champion Pipeline Training**: Train tuned Multinomial Naive Bayes with TF-IDF and n-grams.
6. **Comprehensive Diagnostics**: Assess Accuracy, Precision, Recall, F1, and Confusion Matrix.
7. **Explainable AI (XAI)**: Extract discriminative tokens via log-odds ratios $\log \frac{P(w|\text{Spam})}{P(w|\text{Ham})}$.
8. **Live Threat Inference**: Score real-world phishing lures and legitimate workplace communication.

In [ ]:
# 1. Setup & Imports
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from main import (
    load_spam_dataset,
    train_spam_classifier,
    evaluate_classifier,
    extract_top_features,
    predict_message,
    compare_model_variants
)

print("Modules imported successfully.")

### 2. Dataset Synthesis & Exploratory Analysis
We load a diverse corpus spanning financial fraud, banking phishing, delivery alerts, workplace communications, and personal chats.

In [ ]:
# Load corpus
df = load_spam_dataset(target_size=1200, spam_ratio=0.35, random_seed=42)

print(f"Total Corpus Size: {len(df):,} messages")
print(f"Class Distribution:\n{df['label'].value_counts().rename({0: 'Ham (Legitimate)', 1: 'Spam (Threat)')}")

# Add text statistics
df['char_count'] = df['text'].apply(len)
df['word_count'] = df['text'].apply(lambda t: len(t.split()))

display(df.groupby('label')[['char_count', 'word_count']].mean().rename(index={0: 'Ham', 1: 'Spam'}))
df.head(6)

### 3. Stratified Partitioning
Partition the data into training (75%) and holdout test (25%) splits preserving exact class proportions.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.25, random_state=42, stratify=df['label']
)

print(f"Training instances: {len(X_train):,} ({sum(y_train == 0)} Ham, {sum(y_train == 1)} Spam)")
print(f"Holdout test instances: {len(X_test):,} ({sum(y_test == 0)} Ham, {sum(y_test == 1)} Spam)")

### 4. Model Architecture Benchmarking
Benchmark three Naive Bayes formulations across TF-IDF and Bag-of-Words representations:
- **Multinomial Naive Bayes**: Models discrete term frequency and TF-IDF weights.
- **Complement Naive Bayes**: Corrects for class imbalance by estimating parameters from complement classes.
- **Bernoulli Naive Bayes**: Operates on binary word occurrence indicators.

In [ ]:
benchmark_df = compare_model_variants(X_train, X_test, y_train, y_test)
benchmark_df

### 5. Champion Pipeline Training & Diagnostics
Train the champion **Multinomial Naive Bayes** pipeline configured with sublinear TF-IDF scaling, unigram + bigram feature extraction, and Lidstone smoothing ($\alpha = 0.50$).

In [ ]:
champion_pipe = train_spam_classifier(
    X_train, y_train,
    model_type="multinomial",
    vectorizer_type="tfidf",
    alpha=0.5,
    ngram_range=(1, 2)
)

eval_res = evaluate_classifier(champion_pipe, X_test, y_test, threshold=0.50)

print("=== CHAMPION MODEL HOLDOUT EVALUATION ===")
print(f"Accuracy:  {eval_res['accuracy'] * 100:.2f}%")
print(f"Precision: {eval_res['precision'] * 100:.2f}%")
print(f"Recall:    {eval_res['recall'] * 100:.2f}%")
print(f"F1-Score:  {eval_res['f1'] * 100:.2f}%")
print(f"ROC-AUC:   {eval_res['roc_auc']:.4f}\n")

print("Classification Report:")
print(eval_res['classification_report'])

print("Confusion Matrix:")
print(eval_res['confusion_matrix'])

### 6. Explainable AI: Token Log-Odds Attribution
Extract which words most aggressively shift class likelihood toward Spam or Ham based on the log-odds ratio:
$$
\text{Log-Odds}(w) = \log P(w \mid \text{Spam}) - \log P(w \mid \text{Ham})
$$

In [ ]:
top_spam, top_ham = extract_top_features(champion_pipe, top_n=10)

df_features = pd.DataFrame({
    "Top Spam Tokens": [f"{t} (+{score:.2f})" for t, score in top_spam],
    "Top Ham Tokens": [f"{t} (+{score:.2f})" for t, score in top_ham]
})
df_features

### 7. Real-World Live Threat Inference
Test incoming messages with diverse threat characteristics and observe model confidence and key signal attribution.

In [ ]:
samples = [
    "URGENT: Your Chase bank account has been suspended. Verify identity at bit.ly/chase-auth to restore funds.",
    "WINNER! You won $1,000 Amazon Gift card voucher. Claim immediately at http://amzn-gift-card.xyz",
    "Hey Sarah, let's schedule our design sprint review meeting for Thursday at 10:30 AM in Room 3B.",
    "USPS Tracking: Package #US9827419 has an unpaid postage fee of $2.85. Settle now to receive parcel.",
    "Can you pick up some milk and fresh bread on your way back home from work?"
]

for s in samples:
    res = predict_message(champion_pipe, s, threshold=0.50)
    status = "SPAM" if res['is_spam'] else "HAM (SAFE)"
    signals = ", ".join([f"{t} ({sc:+.1f})" for t, sc in res['top_contributing_tokens']])
    print(f"[{status}] P(Spam)={res['spam_probability']*100:.1f}%")
    print(f"  Text: '{s}'")
    print(f"  Signals: {signals}\n")